# Experiment E1 on Colab: zero-shot transfer of J4 to 3x3 and 2x2 through the mapping (M1, M3)

Runtime -> Change runtime type -> **GPU**, then run all cells. It reads the Step 4 checkpoints from the same Drive
folder as `step4_colab.ipynb` (`MyDrive/Ruba_step4/saved_models/cube4/snapshots/itr_<N>`) and writes everything to
`MyDrive/Ruba_step4/results/e1/`.

* **What runs:** for each checkpoint, embedded 3x3 and 2x2 scrambles (50 per depth, depths 1-15, the same seed-0
  scrambles as the Step 4 evals) are solved with the Step 3 solver: weighted A* (weight 0.8) over the 4x4 embedding
  with J4. Baselines under the same per-cube node budget: the same A* with h = 0 (breadth-first search) and a random
  policy (random walks of `RANDOM_MAX_LEN` moves, restarted until the budget is spent).
* **Heuristic scale:** a 2x2 move is two 4x4 moves, so A* on the 2x2 uses J4 / 2 (`H_SCALE_2X2 = 0.5`); the 3x3 uses
  J4 as is. Beam search (optional) ranks states, so it uses J4 unscaled.
* **Node budget:** `NODE_BUDGET` is shared by every method. A* checks it between steps, so a failing cube can go up to
  `batch_size x 12` nodes over (the same as in Step 4).
* **Resumable:** results are saved after every (target, solver, depth) section; re-running skips finished sections
  as long as the settings are unchanged. If the session dies, just run all cells again.
* **Cost:** from the Step 5 timings, about 10-20 min of GPU per checkpoint and about 40 min (CPU only) for the
  baselines, which run once.
* **Outputs:** `baselines.json`, `itr_<N>.json` (per cube: scramble, path, nodes, time), `summary.json` (one row per
  checkpoint/baseline, target, solver and depth: solve rate, Wilson 95% CI, nodes, solution length) and three plots:
  `e1_by_depth.png`, `e1_vs_checkpoint.png`, `e1_3x3_vs_2x2.png`.
* If the repository is private, set `REPO_URL` to `https://<token>@github.com/bobcoelau12/Ruba.git` and do not save
  the notebook with the token in it.

In [ ]:
# ---- configuration ----
REPO_URL = "https://github.com/bobcoelau12/Ruba.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/Ruba_step4"     # same Drive folder as step4_colab.ipynb
REPO_DIR = "/content/Ruba"

SNAPSHOTS_DIR = f"{DRIVE_ROOT}/saved_models/cube4/snapshots"
OUT_DIR = f"{DRIVE_ROOT}/results/e1"
STEP4_RESULTS = f"{DRIVE_ROOT}/results"              # Step 4 eval_itr_<N>.json files (reused where identical)

# Checkpoints: None = every itr_<N> snapshot on Drive. Or a list, e.g. [16000, 54600, 109200, 163800].
# "current" adds the latest training checkpoint (saved_models/cube4/current, e.g. itr 167400) if it is not a snapshot.
CHECKPOINTS = None

TARGETS = ["embedded3", "embedded2"]   # add "native4" for a native 4x4 reference
SOLVERS = ["astar"]                    # the Step 3 solver; add "beam" to also run beam search with J4
BASELINES = ["bfs", "random"]
DEPTHS = [1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 15]
NUM_PER_DEPTH = 50
NODE_BUDGET = 200000                   # per cube, shared by every method
WEIGHT = 0.8                           # A*: cost = WEIGHT * g + scale * J4
BATCH_SIZE = 100                       # A*: nodes popped per step
H_SCALE_3X3 = 1.0
H_SCALE_2X2 = 0.5                      # J4 is about 2x the 2x2 distance
RANDOM_MAX_LEN = 30                    # random policy: moves per walk before restarting
RANDOM_WALKERS = 1000                  # random policy: walks run in parallel
BEAM_WIDTH, BEAM_MAX_STEPS = 256, 50   # only used if "beam" in SOLVERS

E1_ARGS = (f"--snapshots_dir {SNAPSHOTS_DIR} --out_dir {OUT_DIR} "
           f"--targets {' '.join(TARGETS)} --solvers {' '.join(SOLVERS)} --baselines {' '.join(BASELINES)} "
           f"--depths {' '.join(map(str, DEPTHS))} --num_per_depth {NUM_PER_DEPTH} --max_nodes {NODE_BUDGET} "
           f"--weight {WEIGHT} --batch_size {BATCH_SIZE} --h_scale_3x3 {H_SCALE_3X3} --h_scale_2x2 {H_SCALE_2X2} "
           f"--random_max_len {RANDOM_MAX_LEN} --random_walkers {RANDOM_WALKERS} "
           f"--beam_width {BEAM_WIDTH} --beam_max_steps {BEAM_MAX_STEPS} --reuse_step4_dir {STEP4_RESULTS}")
if CHECKPOINTS:
    E1_ARGS += " --checkpoints " + " ".join(str(c) for c in CHECKPOINTS)
print(E1_ARGS)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.isdir(SNAPSHOTS_DIR), f"no snapshots at {SNAPSHOTS_DIR}: check DRIVE_ROOT"
print("snapshots on Drive:", sorted(os.listdir(SNAPSHOTS_DIR), key=lambda n: int(n.split("_")[-1]) if n.split("_")[-1].isdigit() else 0))
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import os, subprocess
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
assert os.path.isfile(f"{REPO_DIR}/scripts/e1_transfer.py"), "this commit has no scripts/e1_transfer.py: check BRANCH"

In [ ]:
# Evaluation does not need jumanji; skipping it keeps Colab's own JAX untouched.
%cd {REPO_DIR}
!grep -v jumanji requirements.txt > /tmp/requirements_colab.txt && pip install -q -r /tmp/requirements_colab.txt

In [ ]:
# DeepCubeA only uses the GPU when CUDA_VISIBLE_DEVICES is set; otherwise it silently runs on CPU.
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONPATH"] = REPO_DIR
sys.path.insert(0, REPO_DIR)

import torch
from utils import nnet_utils
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
device, devices, on_gpu = nnet_utils.get_device()
print("device:", device, "devices:", devices, "on_gpu:", on_gpu, "-", torch.cuda.get_device_name(0))
assert on_gpu and device.type == "cuda", "DeepCubeA would fall back to CPU"

## Run E1
Baselines first (once, no network), then every checkpoint. Plots and `summary.json` are refreshed after each
checkpoint, so partial results are usable if the session ends. Re-run this cell to resume.

In [ ]:
!python -u scripts/e1_transfer.py {E1_ARGS}

## Results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

summary = json.load(open(f"{OUT_DIR}/summary.json"))
df = pd.DataFrame(summary["rows"])
df["solved/n"] = df["solved"].astype(str) + "/" + df["n"].astype(str)
df["checkpoint"] = df["source"]
cols = ["target", "solver", "checkpoint", "depth", "solved/n", "ci_low", "ci_high", "mean_len", "mean_excess_len",
        "mean_nodes", "median_nodes", "budget_hits"]
pd.set_option("display.max_rows", 500)

# solve rate by depth: one column per checkpoint / baseline
for target in TARGETS:
    t = df[df.target == target].copy()
    t["method"] = t.apply(lambda r: f"{r.solver} {r.source}" if r.source != "baseline" else r.solver, axis=1)
    print(f"\n== {target}: solved / {NUM_PER_DEPTH} ==")
    display(t.pivot_table(index="depth", columns="method", values="solved", aggfunc="first"))
    print(f"== {target}: mean nodes per cube ==")
    display(t.pivot_table(index="depth", columns="method", values="mean_nodes", aggfunc="first").round(0))
    print(f"== {target}: mean solution length (solved cubes) ==")
    display(t.pivot_table(index="depth", columns="method", values="mean_len", aggfunc="first").round(2))
print("replay failures:", int(df["found_but_replay_failed"].sum()))

In [ ]:
for f in ("e1_by_depth.png", "e1_vs_checkpoint.png", "e1_3x3_vs_2x2.png"):
    p = f"{OUT_DIR}/{f}"
    if os.path.isfile(p):
        print(f); display(Image(p))

In [ ]:
# Full table (every row of summary.json); also saved as CSV next to the JSON files
df[cols].to_csv(f"{OUT_DIR}/summary.csv", index=False)
display(df[cols])
print("files in", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))